# AI/ML Project : Hotel Booking Cancellation
### Starter Notebook

Build a model that predicts the **probability that each booking will be cancelled**, then generate `submission.csv` for `test.csv`.

Work through the steps below in order and write your own code in new cells under each step. Only the last cell is provided: it generates and validates your submission file.

**Data location:** `/kaggle/input/ai-ml-project-hotel-booking-cancellation/` (files: `train.csv`, `test.csv`, `sample_submission.csv`)

**Rules:** use only the files provided in this competition. Submit probabilities between 0 and 1, not 0/1 labels.

## Step 1: Load and explore the data
Load `train.csv` and `test.csv`. Understand what is in them before you build anything.

In [1]:
# Step 1: Load and explore
import os, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

SEED = 42
ID, TARGET = "booking_id", "is_canceled"
DATA_DIR = "/kaggle/input/ai-ml-project-hotel-booking-cancellation"
if not os.path.exists(DATA_DIR):
    # Running on your own computer: use a folder called "data" next to this notebook.
    # If it isn't there yet, find train.csv + test.csv (Downloads, Desktop, Documents, OneDrive)
    # and copy them into it automatically.
    import glob, shutil
    DATA_DIR = "data"
    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        home = os.path.expanduser("~")
        found = None
        for place in [".", "Downloads", "Desktop", "Documents", "OneDrive"]:
            for tr_path in glob.glob(os.path.join(home if place != "." else ".", place, "**", "train*.csv"), recursive=True):
                folder = os.path.dirname(tr_path)
                te = glob.glob(os.path.join(folder, "test*.csv"))
                if te and "booking_id" in open(tr_path).readline() and "booking_id" in open(te[0]).readline():
                    found = (tr_path, te[0]); break
            if found: break
        if found is None:
            raise FileNotFoundError("Couldn't find train.csv and test.csv. Download them from the "
                                    "competition's Data tab and put them in a folder called 'data' "
                                    "next to this notebook.")
        os.makedirs(DATA_DIR, exist_ok=True)
        shutil.copy(found[0], os.path.join(DATA_DIR, "train.csv"))
        shutil.copy(found[1], os.path.join(DATA_DIR, "test.csv"))
        print("Copied data from:", os.path.dirname(os.path.abspath(found[0])), "->", os.path.abspath(DATA_DIR))

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), low_memory=False)
print("train:", train.shape, "| test:", test.shape)
print(f"Cancellation rate: {train[TARGET].mean():.1%}  (imbalanced: ~28% cancelled)")
train.head()

Copied data from: C:\Users\Jessica\Downloads -> c:\Users\Jessica\OneDrive\Desktop\data
train: (68121, 28) | test: (17031, 27)
Cancellation rate: 27.8%  (imbalanced: ~28% cancelled)


,booking_id,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,total_of_special_requests,is_canceled
0,1,Resort Hotel,0,2017,August,34,21,0,2,1,0.0,0,BB,ITA,Online TA,TA/TO,0,0,0,E,No Deposit,9.0,NaN,0,Transient,90.63,1,0
1,2,City Hotel,24,2016,January,3,23,0,3,1,0.0,0,BB,GBR,Online TA,Corporate,0,0,0,A,No Deposit,240.0,40.0,0,Transient,85.50,1,0
2,4,Resort Hotel,19,2017,June,26,30,0,2,2,0.0,0,BB,GNB,Online TA,TA/TO,0,0,0,A,No Deposit,250.0,NaN,0,Transient,55.44,1,0
3,5,Resort Hotel,120,2017,May,19,11,2,4,2,0.0,0,BB,AGO,Online TA,TA/TO,0,0,0,A,No Deposit,9.0,NaN,0,Transient-Party,200.70,2,0
4,6,Resort Hotel,29,2016,January,3,21,0,2,1,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,D,No Deposit,250.0,148.0,0,Transient,104.20,0,0


In [2]:
# What is missing, and what looks wrong?
print("Missing values (train):\n", train.isna().sum()[lambda s: s > 0], "\n")
print("Missing values (test):\n", test.isna().sum()[lambda s: s > 0], "\n")
print("Suspicious values in train:")
print("  adr (price/night) max:", train["adr"].max(), "| adr < 0:", (train["adr"] < 0).sum())
print("  adults > 10:", (train["adults"] > 10).sum(), "| babies >= 10:", (train["babies"] >= 10).sum(),
      "| children >= 10:", (train["children"] >= 10).sum())
print("  bookings with 0 guests:", ((train["adults"] + train["children"].fillna(0) + train["babies"]) == 0).sum())
print("  bookings with 0 nights:", ((train["stays_in_weekend_nights"] + train["stays_in_week_nights"]) == 0).sum())
for c in ["meal", "market_segment", "distribution_channel"]:
    print(f"  '{c}' = Undefined:", (train[c] == "Undefined").sum())

# Cancellation rate by the most telling columns
for c in ["deposit_type", "market_segment", "customer_type", "hotel"]:
    print("\n", train.groupby(c)[TARGET].agg(["mean", "size"]).round(3).sort_values("mean"))

Missing values (train):
 children        3
country       362
agent        9159
company     64095
dtype: int64 

Missing values (test):
 country       90
agent       2285
company    16021
dtype: int64 

Suspicious values in train:
  adr (price/night) max: 5400.0 | adr < 0: 0
  adults > 10: 7 | babies >= 10: 1 | children >= 10: 1
  bookings with 0 guests: 267
  bookings with 0 nights: 3115
  'meal' = Undefined: 346
  'market_segment' = Undefined: 3
  'distribution_channel' = Undefined: 5

                mean   size
deposit_type              
No Deposit    0.271  67291
Refundable    0.591     66
Non Refund    0.785    764

                  mean   size
market_segment              
Undefined       0.000      3
Offline TA/TO   0.180  10635
Corporate       0.190   3188
Complementary   0.199    592
Direct          0.201   9389
Aviation        0.231    186
Groups          0.242   3395
Online TA       0.332  40733

                   mean   size
customer_type                
Transient-Party  0

## Step 2: Prepare the data
Clean the data and handle anything that is missing or does not make sense. Apply the same steps to both `train` and `test`, and keep every row of `test`.

In [3]:
# Step 2: Prepare the data (same function for train and test, keeps every test row)
MONTHS = ["january", "february", "march", "april", "may", "june", "july",
          "august", "september", "october", "november", "december"]
MONTH_MAP = {m: i + 1 for i, m in enumerate(MONTHS)}
MONTH_MAP.update({m[:3]: i + 1 for i, m in enumerate(MONTHS)})
NUMERIC = ["lead_time", "arrival_date_year", "arrival_date_week_number", "arrival_date_day_of_month",
           "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
           "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
           "agent", "company", "days_in_waiting_list", "adr", "total_of_special_requests"]
MISSING = {"": np.nan, "nan": np.nan, "none": np.nan, "null": np.nan, "na": np.nan, "undefined": np.nan}

def clean(df):
    df = df.copy()
    # month name -> number (also handles "jul", " JULY ", "7")
    m = df["arrival_date_month"].astype(str).str.strip().str.lower()
    df["arrival_date_month"] = m.map(MONTH_MAP).fillna(pd.to_numeric(m, errors="coerce"))
    # numbers stored as text -> numbers
    for c in NUMERIC:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # text: trim, one case, and treat "Undefined" as missing
    for c in df.columns:
        if c not in (ID, TARGET) and not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.lower().replace(MISSING)
    df["meal"] = df["meal"].fillna("sc")               # Undefined meal = no meal package (SC)
    df["country"] = df["country"].fillna("unknown")
    df["children"] = df["children"].fillna(0)
    df["agent"] = df["agent"].fillna(0)               # missing agent = booked directly
    df["company"] = df["company"].fillna(0)           # missing company = private guest
    # impossible / extreme values -> NaN (LightGBM learns what to do with NaN)
    for c in ["lead_time", "adr", "adults", "children", "babies", "stays_in_week_nights", "stays_in_weekend_nights"]:
        df.loc[df[c] < 0, c] = np.nan
    df.loc[df["adr"] > 1000, "adr"] = np.nan          # the 5400/night booking is a typo
    df.loc[df["adults"] > 10, "adults"] = 10           # cap huge group sizes
    df.loc[df["children"] > 5, "children"] = np.nan    # 10 children / 10 babies are data errors
    df.loc[df["babies"] > 5, "babies"] = np.nan
    return df

train = clean(train)
test = clean(test)
assert len(test) == 17031, "test must keep every row"
print("Missing after cleaning (train):\n", train.isna().sum()[lambda s: s > 0])

Missing after cleaning (train):
 children                1
babies                  1
market_segment          3
distribution_channel    5
adr                     2
dtype: int64


## Step 3: Build features
Turn the columns into numbers your model can learn from, and create new features that help it. Make sure `train` and `test` are encoded the same way.

In [4]:
# Step 3: Build features (train and test encoded together, so categories match)
def add_features(df):
    df = df.copy()
    arrival = pd.to_datetime(dict(year=df["arrival_date_year"], month=df["arrival_date_month"],
                                  day=df["arrival_date_day_of_month"]), errors="coerce")
    booked = arrival - pd.to_timedelta(df["lead_time"], unit="D")
    nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    guests = df["adults"].fillna(0) + df["children"].fillna(0) + df["babies"].fillna(0)
    prev = df["previous_cancellations"] + df["previous_bookings_not_canceled"]

    df["arrival_dow"] = arrival.dt.dayofweek               # day of week of arrival
    df["booking_month"] = booked.dt.month                   # when the booking was made
    df["booking_dow"] = booked.dt.dayofweek
    df["arrival_ordinal"] = (arrival - pd.Timestamp("2015-01-01")).dt.days
    df["total_nights"] = nights
    df["weekend_share"] = df["stays_in_weekend_nights"] / nights.replace(0, np.nan)
    df["total_guests"] = guests
    df["has_kids"] = ((df["children"].fillna(0) + df["babies"].fillna(0)) > 0).astype(int)
    df["no_guests"] = (guests == 0).astype(int)
    df["zero_nights"] = (nights == 0).astype(int)
    df["adr_per_person"] = df["adr"] / guests.replace(0, np.nan)
    df["total_cost"] = df["adr"] * nights
    df["prev_total_bookings"] = prev
    df["prev_cancel_rate"] = df["previous_cancellations"] / (prev + 1)
    df["has_agent"] = (df["agent"] > 0).astype(int)
    df["has_company"] = (df["company"] > 0).astype(int)
    df["lead_per_night"] = df["lead_time"] / (nights + 1)
    df["waited"] = (df["days_in_waiting_list"] > 0).astype(int)
    return df

train = add_features(train)
test = add_features(test)

# Text columns + agent/company IDs are categories, not quantities
CAT_COLS = [c for c in train.columns if c not in (ID, TARGET)
            and not pd.api.types.is_numeric_dtype(train[c])] + ["agent", "company"]

full = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
for c in CAT_COLS:
    full[c] = full[c].fillna("missing").astype(str)
    counts = full[c].value_counts()
    full[c + "_freq"] = full[c].map(counts)                     # how common is this value
    rare = counts[counts < 30].index                             # group very rare values
    full[c] = full[c].where(~full[c].isin(rare), "rare").astype("category")

train_X = full.iloc[:len(train)].reset_index(drop=True)
test_X = full.iloc[len(train):].reset_index(drop=True)
y = train[TARGET].astype(int).values
FEATURES = [c for c in train_X.columns if c != ID]
print(len(FEATURES), "features | categorical:", CAT_COLS)

54 features | categorical: ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type', 'agent', 'company']


## Step 4: Train and validate your model
Hold out part of `train` to check how well your model does on data it has not seen. Compare ideas using ROC-AUC.

In [5]:
%pip install catboost scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Step 4: Train and validate - 5-fold cross-validation, LightGBM + CatBoost
# Identical bookings are kept in the same fold, so validation can't "cheat".
# Class imbalance needs no re-weighting: ROC-AUC only depends on the ranking.
from catboost import CatBoostClassifier
from scipy.stats import rankdata

groups = pd.util.hash_pandas_object(train.drop(columns=[ID, TARGET]), index=False).values
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(train_X, y, groups))

LGB_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.02, n_estimators=5000,
                  num_leaves=127, min_child_samples=20, feature_fraction=0.6,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=3,
                  cat_smooth=50, cat_l2=20, min_data_per_group=50, max_cat_to_onehot=8,
                  random_state=SEED, verbose=-1)

cat_feats = [c for c in CAT_COLS if c in FEATURES]
train_C, test_C = train_X[FEATURES].copy(), test_X[FEATURES].copy()
for c in cat_feats:                                  # CatBoost wants categories as text
    train_C[c] = train_C[c].astype(str).fillna("missing")
    test_C[c] = test_C[c].astype(str).fillna("missing")

oof_lgb, oof_cat = np.zeros(len(y)), np.zeros(len(y))
test_lgb, test_cat = np.zeros(len(test_X)), np.zeros(len(test_X))

for k, (tr, va) in enumerate(folds, 1):
    m = lgb.LGBMClassifier(**LGB_PARAMS)
    m.fit(train_X.loc[tr, FEATURES], y[tr], eval_set=[(train_X.loc[va, FEATURES], y[va])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_lgb[va] = m.predict_proba(train_X.loc[va, FEATURES])[:, 1]
    test_lgb += m.predict_proba(test_X[FEATURES])[:, 1] / len(folds)

    c = CatBoostClassifier(iterations=3000, learning_rate=0.06, depth=8, eval_metric="AUC",
                           od_type="Iter", od_wait=200, random_seed=SEED, verbose=0)
    c.fit(train_C.iloc[tr], y[tr], cat_features=cat_feats, eval_set=(train_C.iloc[va], y[va]),
          use_best_model=True)
    oof_cat[va] = c.predict_proba(train_C.iloc[va])[:, 1]
    test_cat += c.predict_proba(test_C)[:, 1] / len(folds)

    print(f"Fold {k}: LightGBM {roc_auc_score(y[va], oof_lgb[va]):.4f} | "
          f"CatBoost {roc_auc_score(y[va], oof_cat[va]):.4f}")

def blend(a, b, w=0.7):                              # rank average: only order matters for AUC
    return (w * rankdata(a) + (1 - w) * rankdata(b)) / len(a)

print(f"\nLightGBM CV AUC : {roc_auc_score(y, oof_lgb):.5f}")
print(f"CatBoost CV AUC : {roc_auc_score(y, oof_cat):.5f}")
print(f"Blend    CV AUC : {roc_auc_score(y, blend(oof_lgb, oof_cat)):.5f}")

Fold 1: LightGBM 0.8787 | CatBoost 0.8774
Fold 2: LightGBM 0.8776 | CatBoost 0.8753
Fold 3: LightGBM 0.8767 | CatBoost 0.8754
Fold 4: LightGBM 0.8817 | CatBoost 0.8795
Fold 5: LightGBM 0.8833 | CatBoost 0.8822

LightGBM CV AUC : 0.87954
CatBoost CV AUC : 0.87792
Blend    CV AUC : 0.88027


In [7]:
# Which features matter most? (LightGBM, last fold)
imp = pd.Series(m.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
imp.head(15).round(0)

country                      67418.0
lead_time                    65075.0
agent_freq                   52227.0
lead_per_night               39785.0
total_of_special_requests    36554.0
market_segment               35931.0
country_freq                 35892.0
arrival_ordinal              32528.0
agent                        30819.0
total_cost                   21754.0
adr_per_person               17132.0
adr                          17112.0
arrival_date_year            16341.0
customer_type                16300.0
previous_cancellations       15715.0
dtype: float64

## Step 5: Predict on the test set
Train your final model and predict the probability of cancellation for every row in `test`.

Store the result in a variable named **`test_pred`**: one probability between 0 and 1 for each row of `test`, in the same order as `test.csv`.

In [8]:
# Step 5: Predict on the test set
# The final prediction = average of the 5 fold models, LightGBM and CatBoost blended.
test_pred = blend(test_lgb, test_cat)
test_pred = np.clip(test_pred, 0, 1)
print(len(test_pred), "predictions | min", test_pred.min().round(4), "| max", test_pred.max().round(4))

17031 predictions | min 0.0001 | max 0.9999


## Step 6: Generate and validate your submission
Run the cell below. It needs `test_pred` from Step 5, creates `submission.csv`, and checks that it is valid.

In [9]:
# ---- Generate and validate submission.csv (do not change) ----
import os
import numpy as np
import pandas as pd

ID_COL, TARGET = "booking_id", "is_canceled"
input_dir = "/kaggle/input/ai-ml-project-hotel-booking-cancellation"
if not os.path.exists(input_dir):
    input_dir = "data"                                   # fallback outside Kaggle
output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

test_ids = pd.read_csv(os.path.join(input_dir, "test.csv"), usecols=[ID_COL])[ID_COL]
preds = np.asarray(test_pred, dtype=float).ravel()

submission = pd.DataFrame({ID_COL: test_ids.values, TARGET: preds})
path = os.path.join(output_dir, "submission.csv")
submission.to_csv(path, index=False)

# ---- Checks ----
problems = []
if len(preds) != len(test_ids):
    problems.append(f"test_pred has {len(preds)} values, expected {len(test_ids)} (one per test row)")
if np.isnan(preds).any():
    problems.append("test_pred contains missing values")
elif preds.min() < 0 or preds.max() > 1:
    problems.append("test_pred must be probabilities between 0 and 1")
elif len(np.unique(preds)) <= 2:
    problems.append("test_pred looks like 0/1 labels. Submit probabilities instead")

if problems:
    print("SUBMISSION NOT VALID:")
    for p in problems:
        print(" -", p)
else:
    print("Saved:", path, "| shape:", submission.shape)
    print("Submission is valid. Save a version with Save & Run All, then submit submission.csv from the Output tab.")
submission.head()

Saved: .\submission.csv | shape: (17031, 2)
Submission is valid. Save a version with Save & Run All, then submit submission.csv from the Output tab.


,booking_id,is_canceled
0,3,0.051764
1,12,0.518261
2,13,0.684141
3,17,0.728601
4,19,0.543562
